# AI Rock-Paper-Scissors (Webcam Edition, Jupyter version)

Play rock-paper-scissors against the computer using hand gestures read live
from your webcam, right inside a Jupyter notebook.

**Uses MediaPipe's legacy "Solutions" API** (`mp.solutions.hands`), pinned
to `mediapipe==0.10.14` (see `requirements.txt`). We tried the newer Tasks
API first, but it crashed on macOS with a native error trying to init a
Metal/GPU helper even with CPU delegate requested -- a real bug in recent
MediaPipe releases. The Solutions API uses a CPU-only graph and never
touches Metal/GPU at all, and it's the version almost every hand-tracking
tutorial online is built on, so it's extremely battle-tested. Bonus: it
bundles its own model, so there's no separate model file to download.

**How to use this notebook:**
1. Run every cell from top to bottom, **in order** (Cell -> Run All, or step
   through with Shift+Enter). The setup cells just define functions -- nothing
   happens until you run the **"Play!"** cell near the bottom.
2. A native camera window will pop up (outside the browser tab) when you run
   the "Play!" cell. Press **q** in that window to quit at any time.
3. **This only works with a local Jupyter kernel** (classic Notebook,
   JupyterLab, or VS Code's Jupyter extension, all running on your own
   machine) -- it needs to open a real OS window and access your webcam, so
   it will NOT work on a remote/cloud Jupyter server or Google Colab.

If a cell ever gets stuck or the camera won't open on a later run, use the
**"Emergency cleanup"** cell at the very end.


## 1. Imports

In [1]:
import random
import time

import cv2
import mediapipe as mp

print("Imports OK. mediapipe version:", mp.__version__, "| opencv version:", cv2.__version__)
assert mp.__version__.startswith("0.10."), (
    "Expected mediapipe 0.10.x (this notebook needs mp.solutions, which was "
    "removed in 0.10.35+ and the 1.0.x line). Run: pip install \"mediapipe==0.10.14\""
)
assert hasattr(mp, "solutions"), "mp.solutions is missing -- see the assertion message above."
print("mp.solutions is available. Good to go.")

Imports OK. mediapipe version: 0.10.14 | opencv version: 5.0.0
mp.solutions is available. Good to go.


## 2. Constants

In [2]:
CHOICES = ["rock", "paper", "scissors"]
BEATS = {"rock": "scissors", "paper": "rock", "scissors": "paper"}

## 3. Gesture classification

Pure logic -- turns 21 hand landmarks into `"rock"` / `"paper"` / `"scissors"` / `"unknown"`. No camera or mediapipe calls in here, so it's easy to sanity-check on its own (see the self-test cell below).

In [3]:
# MediaPipe hand landmark indices for the four fingers we use. We
# deliberately ignore the thumb -- its motion is mostly horizontal rather
# than vertical, which makes a simple "tip above pip = extended" rule
# unreliable for it, and we don't need it to tell rock/paper/scissors apart.
FINGER_JOINTS = {
    "index": (6, 8),    # (PIP joint index, TIP index)
    "middle": (10, 12),
    "ring": (14, 16),
    "pinky": (18, 20),
}


def finger_states(landmarks) -> dict:
    """Return {finger_name: True/False} for whether each finger is extended."""
    states = {}
    for name, (pip_idx, tip_idx) in FINGER_JOINTS.items():
        pip_y = landmarks[pip_idx].y
        tip_y = landmarks[tip_idx].y
        states[name] = tip_y < pip_y
    return states


def classify_gesture(landmarks) -> str:
    """Classify a hand pose as 'rock', 'paper', 'scissors', or 'unknown'."""
    states = finger_states(landmarks)
    extended = [name for name, is_up in states.items() if is_up]
    count = len(extended)

    if count == 0:
        return "rock"
    if count == 4:
        return "paper"
    if count == 2 and "index" in extended and "middle" in extended:
        return "scissors"
    return "unknown"

### Self-test (optional but recommended)

Run this to confirm the classification logic works correctly *before* touching the webcam -- uses fake landmark data, no camera or model needed.

In [4]:
from dataclasses import dataclass


@dataclass
class _L:
    x: float
    y: float


def _make_landmarks(index_up, middle_up, ring_up, pinky_up):
    lm = [_L(0.5, 0.5) for _ in range(21)]
    pairs = {
        "index": (6, 8, index_up),
        "middle": (10, 12, middle_up),
        "ring": (14, 16, ring_up),
        "pinky": (18, 20, pinky_up),
    }
    for pip_idx, tip_idx, is_up in pairs.values():
        if is_up:
            lm[pip_idx] = _L(0.5, 0.6)
            lm[tip_idx] = _L(0.5, 0.3)
        else:
            lm[pip_idx] = _L(0.5, 0.3)
            lm[tip_idx] = _L(0.5, 0.6)
    return lm


assert classify_gesture(_make_landmarks(False, False, False, False)) == "rock"
assert classify_gesture(_make_landmarks(True, True, True, True)) == "paper"
assert classify_gesture(_make_landmarks(True, True, False, False)) == "scissors"
assert classify_gesture(_make_landmarks(True, False, False, True)) == "unknown"
print("All gesture self-tests passed ✅")

All gesture self-tests passed ✅


## 4. Hand detector setup

Uses `mp.solutions.hands` -- CPU-only under the hood, no Metal/GPU involved, no model download needed (it's bundled).

In [5]:
def create_hands():
    mp_hands = mp.solutions.hands
    mp_draw = mp.solutions.drawing_utils
    hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7, min_tracking_confidence=0.5)
    return hands, mp_hands, mp_draw


print("Detector helper ready.")

Detector helper ready.


## 5. Game loop helpers

In [6]:
def decide_winner(player: str, computer: str) -> str:
    if player == computer:
        return "Tie!"
    if BEATS.get(player) == computer:
        return "You win!"
    return "Computer wins!"


def draw_text(frame, text, y, scale=1.0, color=(255, 255, 255), thickness=2):
    cv2.putText(frame, text, (30, y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness, cv2.LINE_AA)


def run_countdown(cap, hands, mp_hands, mp_draw, player_score, computer_score):
    """Show a 3-2-1-SHOOT countdown and return the gesture captured at the end."""
    sequence = ["3", "2", "1", "SHOOT!"]
    start = time.time()
    captured = "unknown"

    while True:
        ok, frame = cap.read()
        if not ok:
            return "unknown"
        frame = cv2.flip(frame, 1)

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)
        landmarks = None
        if result.multi_hand_landmarks:
            hand = result.multi_hand_landmarks[0]
            landmarks = hand.landmark
            mp_draw.draw_landmarks(frame, hand, mp_hands.HAND_CONNECTIONS)

        step = int(time.time() - start)
        if step < len(sequence):
            draw_text(frame, sequence[step], 110, scale=3, color=(0, 255, 255), thickness=4)
        else:
            if landmarks is not None:
                captured = classify_gesture(landmarks)
            draw_text(frame, "You: %d  Computer: %d" % (player_score, computer_score), 450, scale=0.8)
            cv2.imshow("AI Rock Paper Scissors", frame)
            cv2.waitKey(1)
            return captured

        draw_text(frame, "You: %d  Computer: %d" % (player_score, computer_score), 450, scale=0.8)
        cv2.imshow("AI Rock Paper Scissors", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            return "quit"


def show_result(cap, result_text, player_score, computer_score):
    """Display the round result for up to 3 seconds, or until a key is pressed."""
    start = time.time()
    while time.time() - start < 3:
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.flip(frame, 1)
        draw_text(frame, result_text, 80, scale=0.85, color=(0, 255, 0))
        draw_text(frame, "You: %d  Computer: %d" % (player_score, computer_score), 450, scale=0.8)
        draw_text(frame, "Press any key for next round, q to quit", 520, scale=0.6, color=(200, 200, 200))
        cv2.imshow("AI Rock Paper Scissors", frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            return "quit"
        if key != 255:
            return "continue"
    return "continue"


print("Game loop helpers ready.")

Game loop helpers ready.


## 6. `play_game()` -- the main loop

Calling this (in the next cell) opens your webcam and starts the game. It cleans up the camera and windows in a `finally` block, so even if something goes wrong mid-round, your webcam won't stay locked.

In [7]:
def play_game():
    hands, mp_hands, mp_draw = create_hands()

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("Could not open webcam. Check camera permissions / device index.")

    player_score = 0
    computer_score = 0

    try:
        while True:
            gesture = run_countdown(cap, hands, mp_hands, mp_draw, player_score, computer_score)
            if gesture == "quit":
                break

            computer_choice = random.choice(CHOICES)
            if gesture == "unknown":
                result_text = "Couldn't read your gesture - no points this round."
            else:
                outcome = decide_winner(gesture, computer_choice)
                result_text = "You: %s | Computer: %s -> %s" % (gesture, computer_choice, outcome)
                if outcome == "You win!":
                    player_score += 1
                elif outcome == "Computer wins!":
                    computer_score += 1

            action = show_result(cap, result_text, player_score, computer_score)
            if action == "quit":
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()
        # On macOS, destroyAllWindows() sometimes needs a few waitKey pumps
        # to actually close the window rather than leaving a frozen frame.
        for _ in range(4):
            cv2.waitKey(1)
        hands.close()
        print("Final score -- You: %d  Computer: %d" % (player_score, computer_score))


print("play_game() is defined. Run the next cell to play!")

play_game() is defined. Run the next cell to play!


## 7. Play!

Run this cell to start. A camera window opens outside the browser -- follow the 3-2-1-SHOOT countdown and show rock, paper, or scissors. Press **q** in that window to stop. You can re-run just this cell to play again.

In [8]:
play_game()

I0000 00:00:1787061035.701622 2539576 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1787061035.719906 2539794 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787061035.725688 2539794 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/Users/pranavmengi/Downloads/rps-ai-game 5/venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Final score -- You: 3  Computer: 4


## Emergency cleanup

Only run this if a previous run left the camera window frozen or the webcam seems stuck/unavailable on a later attempt (e.g. you interrupted the kernel mid-round).

In [9]:
cv2.destroyAllWindows()
for _ in range(4):
    cv2.waitKey(1)
_tmp_cap = cv2.VideoCapture(0)
_tmp_cap.release()
print("Cleaned up. Try running the Play! cell again.")

Cleaned up. Try running the Play! cell again.
